# Module 08: Training - Complete Learning Loops

In [6]:
#| default_exp foundation.training
#| export

import numpy as np
rng = np.random.default_rng(7)
import pickle
import time
from typing import Dict, List, Optional, Tuple, Any, Callable
from pathlib import Path
import sys
import os

# Import dependencies from other modules
from tinytorch.foundation.tensor import Tensor
from tinytorch.foundation.layers import Linear
from tinytorch.foundation.losses import MSELoss, CrossEntropyLoss
from tinytorch.foundation.optimizers import SGD, AdamW

# Enable autograd for gradient tracking (required for training)
from tinytorch.foundation.autograd import enable_autograd
enable_autograd()

# Constants for learning rate scheduling defaults
DEFAULT_MAX_LR = 0.1  # Default maximum learning rate for cosine schedule
DEFAULT_MIN_LR = 0.01  # Default minimum learning rate for cosine schedule
DEFAULT_TOTAL_EPOCHS = 100  # Default total epochs for learning rate schedule

###  Learning Rate Scheduling: Adaptive Training Speed


In [7]:
#| export
class CosineSchedule:
    """
    Cosine annealing learning rate schedule.
    """
    
    def __init__(self, max_lr: float = DEFAULT_MAX_LR, min_lr: float = DEFAULT_MIN_LR, total_epochs: int = DEFAULT_TOTAL_EPOCHS):
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.total_epochs = total_epochs

    def get_lr(self, epoch: int) -> float:
        """Get learning rate for current epoch."""
        if epoch >= self.total_epochs:
            return self.min_lr

        # Cosine annealing formula
        cosine_factor = (1 + np.cos(np.pi * epoch / self.total_epochs)) / 2
        return self.min_lr + (self.max_lr - self.min_lr) * cosine_factor
    

###  Unit Test: CosineSchedule

This test validates our learning rate scheduling implementation.

In [8]:
def test_unit_cosine_schedule():
    """ Test CosineSchedule implementation."""
    print(" Unit Test: CosineSchedule...")

    # Test basic schedule
    schedule = CosineSchedule(max_lr=0.1, min_lr=0.01, total_epochs=100)

    # Test start, middle, and end
    lr_start = schedule.get_lr(0)
    lr_middle = schedule.get_lr(50)
    lr_end = schedule.get_lr(100)

    print(f"Learning rate at epoch 0: {lr_start:.4f}")
    print(f"Learning rate at epoch 50: {lr_middle:.4f}")
    print(f"Learning rate at epoch 100: {lr_end:.4f}")

    # Validate behavior
    assert abs(lr_start - 0.1) < 1e-6, f"Expected 0.1 at start, got {lr_start}"
    assert abs(lr_end - 0.01) < 1e-6, f"Expected 0.01 at end, got {lr_end}"
    assert 0.01 < lr_middle < 0.1, f"Middle LR should be between min and max, got {lr_middle}"

    # Test monotonic decrease in first half
    lr_quarter = schedule.get_lr(25)
    assert lr_quarter > lr_middle, "LR should decrease monotonically in first half"

    print(" CosineSchedule works correctly!")

if __name__ == "__main__":
    test_unit_cosine_schedule()

 Unit Test: CosineSchedule...
Learning rate at epoch 0: 0.1000
Learning rate at epoch 50: 0.0550
Learning rate at epoch 100: 0.0100
 CosineSchedule works correctly!


###  Gradient Clipping: Preventing Training Explosions

In [9]:
#| export

def clip_grad_norm(parameters: List, max_norm: float = 1.0) -> float:
    """
    Clip gradients by global norm to prevent exploding gradients.
    """
    
    if not parameters:
        return 0.0

    # Collect all gradients and compute global norm
    total_norm = 0.0
    for param in parameters:
        if param.grad is not None:
            # Handle both Tensor gradients and numpy array gradients
            if isinstance(param.grad, np.ndarray):
                grad_data = param.grad
            else:
                # Trust that Tensor has .data attribute
                grad_data = param.grad.data
            total_norm += np.sum(grad_data ** 2)

    total_norm = np.sqrt(total_norm)

    # Clip if necessary
    if total_norm > max_norm:
        clip_coef = max_norm / total_norm
        for param in parameters:
            if param.grad is not None:
                # Handle both Tensor gradients and numpy array gradients
                if isinstance(param.grad, np.ndarray):
                    param.grad = param.grad * clip_coef
                else:
                    # Trust that Tensor has .data attribute
                    param.grad.data = param.grad.data * clip_coef

    return float(total_norm)
    

###  Unit Test: Gradient Clipping

This test validates our gradient clipping implementation.

In [10]:
def test_unit_clip_grad_norm():
    """ Test clip_grad_norm implementation."""
    print(" Unit Test: Gradient Clipping...")

    # Use real Tensor from Module 01
    import sys
    # Tensor already imported at module level

    # Test case 1: Large gradients that need clipping
    param1 = Tensor([1.0, 2.0], requires_grad=True)
    param1.grad = np.array([3.0, 4.0])  # norm = 5.0

    param2 = Tensor([3.0, 4.0], requires_grad=True)
    param2.grad = np.array([6.0, 8.0])  # norm = 10.0

    params = [param1, param2]
    # Total norm = sqrt(5² + 10²) = sqrt(125) ≈ 11.18

    original_norm = clip_grad_norm(params, max_norm=1.0)

    # Check original norm was large
    assert original_norm > 1.0, f"Original norm should be > 1.0, got {original_norm}"

    # Check gradients were clipped
    new_norm = 0.0
    for param in params:
        if isinstance(param.grad, np.ndarray):
            grad_data = param.grad
        else:
            # Trust that Tensor has .data attribute
            grad_data = param.grad.data
        new_norm += np.sum(grad_data ** 2)
    new_norm = np.sqrt(new_norm)

    print(f"Original norm: {original_norm:.2f}")
    print(f"Clipped norm: {new_norm:.2f}")

    assert abs(new_norm - 1.0) < 1e-6, f"Clipped norm should be 1.0, got {new_norm}"

    # Test case 2: Small gradients that don't need clipping
    small_param = Tensor([1.0, 2.0], requires_grad=True)
    small_param.grad = np.array([0.1, 0.2])
    small_params = [small_param]
    original_small = clip_grad_norm(small_params, max_norm=1.0)

    assert original_small < 1.0, "Small gradients shouldn't be clipped"

    print("✅ Gradient clipping works correctly!")

if __name__ == "__main__":
    test_unit_clip_grad_norm()

 Unit Test: Gradient Clipping...
Original norm: 11.18
Clipped norm: 1.00
✅ Gradient clipping works correctly!


###  The Trainer Class: Orchestrating Complete Training

The Trainer class coordinates all the components we've built (model, optimizer, loss
function, scheduler) into a unified training system. 

In [11]:
#| export
class Trainer:
    """
    Complete training orchestrator for neural networks.
    """

    def _get_model_state(self):
        """Extract model parameters for checkpointing."""
        return {i: param.data.copy() for i, param in enumerate(self.model.parameters())}

    def _set_model_state(self, state):
        """Restore model parameters from checkpoint."""
        for i, param in enumerate(self.model.parameters()):
            if i in state:
                param.data = state[i].copy()

    def _get_optimizer_state(self):
        """Extract optimizer state for checkpointing."""
        state = {}
        state['lr'] = self.optimizer.lr
        if hasattr(self.optimizer, 'has_momentum') and self.optimizer.has_momentum():
            momentum_state = self.optimizer.get_momentum_state()
            if momentum_state is not None:
                state['momentum_buffers'] = momentum_state
        return state

    def _set_optimizer_state(self, state):
        """Restore optimizer state from checkpoint."""
        if 'lr' in state:
            self.optimizer.lr = state['lr']
        if 'momentum_buffers' in state:
            if hasattr(self.optimizer, 'has_momentum') and self.optimizer.has_momentum():
                self.optimizer.set_momentum_state(state['momentum_buffers'])

    def _get_scheduler_state(self):
        """Extract scheduler state for checkpointing."""
        if self.scheduler is None:
            return None
        return {
            'max_lr': getattr(self.scheduler, 'max_lr', None),
            'min_lr': getattr(self.scheduler, 'min_lr', None),
            'total_epochs': getattr(self.scheduler, 'total_epochs', None)
        }

    def _set_scheduler_state(self, state):
        """Restore scheduler state from checkpoint."""
        if state is None or self.scheduler is None:
            return
        for key, value in state.items():
            if hasattr(self.scheduler, key):
                setattr(self.scheduler, key, value)

### Trainer.__init__ - Setting Up the Training System

The constructor stores all training components and initializes tracking state.

In [13]:
#| export
def trainer_init(self, model, optimizer, loss_fn, scheduler=None, grad_clip_norm=None):
    """
    Initialize trainer with model and training components.
    """
    
    self.model = model
    self.optimizer = optimizer
    self.loss_fn = loss_fn
    self.scheduler = scheduler
    self.grad_clip_norm = grad_clip_norm

    # Enable gradient tracking for all model parameters.
    # Layers (e.g. Linear) may be created without requires_grad=True,
    # so we set it explicitly here to ensure backward() populates param.grad.
    # Guard against raw numpy arrays passed by test stubs or non-Tensor params.
    for param in model.parameters():
        if isinstance(param, Tensor):
            param.requires_grad = True

    # Training state
    self.epoch = 0
    self.step = 0
    self.training_mode = True

    # History tracking
    self.history = {
        'train_loss': [],
        'eval_loss': [],
        'learning_rates': []
    }
    

Trainer.__init__ = trainer_init

###  Unit Test: Trainer.__init__

In [14]:
def test_unit_trainer_init():
    """ Test Trainer.__init__ implementation."""
    print(" Unit Test: Trainer.__init__...")

    class DummyModel:
        def __init__(self):
            self.training = True
        def forward(self, x):
            return x
        def parameters(self):
            return []

    model = DummyModel()
    optimizer = SGD([], lr=0.01)
    loss_fn = MSELoss()
    scheduler = CosineSchedule(max_lr=0.1, min_lr=0.01, total_epochs=10)

    trainer = Trainer(model, optimizer, loss_fn, scheduler, grad_clip_norm=1.0)

    # Verify components stored
    assert trainer.model is model, "Model not stored"
    assert trainer.optimizer is optimizer, "Optimizer not stored"
    assert trainer.loss_fn is loss_fn, "Loss function not stored"
    assert trainer.scheduler is scheduler, "Scheduler not stored"
    assert trainer.grad_clip_norm == 1.0, "Grad clip norm not stored"

    # Verify state initialization
    assert trainer.epoch == 0, f"Expected epoch=0, got {trainer.epoch}"
    assert trainer.step == 0, f"Expected step=0, got {trainer.step}"
    assert trainer.training_mode is True, "Expected training_mode=True"

    # Verify history
    assert 'train_loss' in trainer.history, "Missing train_loss in history"
    assert 'eval_loss' in trainer.history, "Missing eval_loss in history"
    assert 'learning_rates' in trainer.history, "Missing learning_rates in history"
    assert len(trainer.history['train_loss']) == 0, "train_loss should be empty"

    # Test without optional args
    trainer2 = Trainer(model, optimizer, loss_fn)
    assert trainer2.scheduler is None, "Scheduler should default to None"
    assert trainer2.grad_clip_norm is None, "Grad clip should default to None"

    print("✅ Trainer.__init__ works correctly!")

if __name__ == "__main__":
    test_unit_trainer_init()

 Unit Test: Trainer.__init__...
✅ Trainer.__init__ works correctly!


###  Trainer.train_epoch - The Core Learning Loop

Each epoch iterates through the dataset, performing
the forward-backward-update cycle that drives learning.

#### Step 1: Process a Single Batch

The inner loop body: run forward pass, compute loss, and run backward pass
with scaled gradients for accumulation.

In [18]:
#| export
def _trainer_process_batch(self, inputs, targets, accumulation_steps):
    """
    Process one batch: forward pass, loss computation, backward pass.
    """

    # Forward pass
    outputs = self.model.forward(inputs)
    loss = self.loss_fn.forward(outputs, targets)

    # Scale loss for accumulation
    scaled_loss = loss.data/accumulation_steps

    # Backward pass with scaled gradients
    scaled_gradient = np.ones_like(loss.data)/accumulation_steps
    loss.backward(scaled_gradient)

    return float(scaled_loss)
    
Trainer._process_batch = _trainer_process_batch 

#### Step 2: Perform Optimizer Update

When enough gradients have accumulated, clip them (if configured),
step the optimizer, and reset gradients for the next accumulation window.

In [30]:
#| export
def _trainer_optimizer_update(self):
    """
    Clip gradients (if enabled) and step the optimizer.
    """

    if self.grad_clip_norm is not None:
        params = self.model.parameters()
        clip_grad_norm(params, self.grad_clip_norm)
    self.optimizer.step()
    self.optimizer.zero_grad()

Trainer._optimizer_update = _trainer_optimizer_update

#### Step 3: Compose the Full Epoch Loop

In [19]:
#| export
def trainer_train_epoch(self, dataloader, accumulation_steps=1):
    """
    Train for one epoch through the dataset.
    """
    
    self.model.training = True
    self.training_mode = True

    # Update scheduler at the start of each epoch so LR is set before training begins
    if self.scheduler is not None:
        current_lr = self.scheduler.get_lr(self.epoch)
        self.optimizer.lr = current_lr
        self.history['learning_rates'].append(current_lr)

    total_loss = 0.0
    num_batches = 0
    accumulated_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(dataloader):
        accumulated_loss += self._process_batch(inputs, targets, accumulation_steps)

        # Update parameters every accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0:
            self._optimizer_update()
            total_loss += accumulated_loss
            accumulated_loss = 0.0
            num_batches += 1
            self.step += 1

    # Handle remaining accumulated gradients
    if accumulated_loss > 0:
        self._optimizer_update()
        total_loss += accumulated_loss
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    self.history['train_loss'].append(avg_loss)

    self.epoch += 1
    return avg_loss

    

Trainer.train_epoch = trainer_train_epoch

###  Unit Test: Trainer._process_batch

In [20]:
def test_unit_trainer_process_batch():
    """ Test Trainer._process_batch implementation."""
    print(" Unit Test: Trainer._process_batch...")

    class SimpleModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    model = SimpleModel()
    optimizer = SGD(model.parameters(), lr=0.01)
    loss_fn = MSELoss()
    trainer = Trainer(model, optimizer, loss_fn)

    inputs = Tensor([[1.0, 0.5]])
    targets = Tensor([[2.0]])

    scaled_loss = trainer._process_batch(inputs, targets, accumulation_steps=1)

    # Should return a float
    assert isinstance(scaled_loss, float), f"Expected float, got {type(scaled_loss)}"
    # Loss should be non-negative
    assert scaled_loss >= 0, f"Expected non-negative loss, got {scaled_loss}"

    print("✅ Trainer._process_batch works correctly!")

if __name__ == "__main__":
    test_unit_trainer_process_batch()

 Unit Test: Trainer._process_batch...
✅ Trainer._process_batch works correctly!


###  Unit Test: Trainer._optimizer_update

In [21]:
def test_unit_trainer_optimizer_update():
    """ Test Trainer._optimizer_update implementation."""
    print(" Unit Test: Trainer._optimizer_update...")

    class SimpleModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    model = SimpleModel()
    optimizer = SGD(model.parameters(), lr=0.01)
    loss_fn = MSELoss()
    trainer = Trainer(model, optimizer, loss_fn)

    # Do a forward-backward to create gradients
    inputs = Tensor([[1.0, 0.5]])
    targets = Tensor([[2.0]])
    trainer._process_batch(inputs, targets, accumulation_steps=1)

    # Record params before update
    params_before = [p.data.copy() for p in model.parameters()]

    # Perform update
    trainer._optimizer_update()

    # Parameters should have changed
    params_after = [p.data for p in model.parameters()]
    changed = any(not np.allclose(b, a) for b, a in zip(params_before, params_after))
    assert changed, "Parameters should change after optimizer update"

    print("✅ Trainer._optimizer_update works correctly!")

if __name__ == "__main__":
    test_unit_trainer_optimizer_update()

 Unit Test: Trainer._optimizer_update...
✅ Trainer._optimizer_update works correctly!


###  Unit Test: Trainer.train_epoch

In [34]:
def test_unit_trainer_train_epoch():
    """ Test Trainer.train_epoch implementation."""
    print(" Unit Test: Trainer.train_epoch...")

    class SimpleModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    model = SimpleModel()
    optimizer = SGD(model.parameters(), lr=0.01)
    loss_fn = MSELoss()
    trainer = Trainer(model, optimizer, loss_fn)

    dataloader = [
        (Tensor([[1.0, 0.5]]), Tensor([[2.0]])),
        (Tensor([[0.5, 1.0]]), Tensor([[1.5]]))
    ]

    # Train one epoch
    loss = trainer.train_epoch(dataloader)

    # Verify return type
    assert isinstance(loss, (float, np.floating)), f"Expected float loss, got {type(loss)}"

    # Verify epoch incremented
    assert trainer.epoch == 1, f"Expected epoch=1, got {trainer.epoch}"

    # Verify history recorded
    assert len(trainer.history['train_loss']) == 1, "Should have 1 loss recorded"

    # Verify model was in training mode
    assert trainer.training_mode is True, "Should be in training mode"

    # Train another epoch - loss should still be a valid number
    loss2 = trainer.train_epoch(dataloader)
    assert isinstance(loss2, (float, np.floating)), f"Second epoch loss should be float"
    assert trainer.epoch == 2, f"Expected epoch=2, got {trainer.epoch}"
    assert len(trainer.history['train_loss']) == 2, "Should have 2 losses recorded"

    # Test with scheduler
    model2 = SimpleModel()
    optimizer2 = SGD(model2.parameters(), lr=0.1)
    scheduler = CosineSchedule(max_lr=0.1, min_lr=0.01, total_epochs=10)
    trainer2 = Trainer(model2, optimizer2, loss_fn, scheduler=scheduler)

    trainer2.train_epoch(dataloader)
    assert len(trainer2.history['learning_rates']) == 1, "Should record LR with scheduler"

    #Test with gradient clipping
    model3 = SimpleModel()
    optimizer3 = SGD(model3.parameters(), lr=0.01)
    trainer3 = Trainer(model3, optimizer3, loss_fn, grad_clip_norm=0.5)

    loss3 = trainer3.train_epoch(dataloader)
    assert isinstance(loss3, (float, np.floating)), "Training with grad clip should work"

    print(f"  Epoch 1 loss: {loss:.4f}")
    print(f"  Epoch 2 loss: {loss2:.4f}")
    print("✅ Trainer.train_epoch works correctly!")

if __name__ == "__main__":
    test_unit_trainer_train_epoch()

 Unit Test: Trainer.train_epoch...
  Epoch 1 loss: 2.2981
  Epoch 2 loss: 1.9584
✅ Trainer.train_epoch works correctly!


###  Trainer.evaluate - Measuring Model Performance

In [35]:
#| export
def trainer_evaluate(self, dataloader):
    """
    Evaluate model on dataset without updating parameters.

    """
    
    self.model.training = False
    self.training_mode = False

    total_loss = 0.0
    correct = 0
    total = 0
    num_batches = 0

    for inputs, targets in dataloader:
        # Forward pass only
        outputs = self.model.forward(inputs)
        loss = self.loss_fn.forward(outputs, targets)

        total_loss += loss.data
        num_batches += 1

        # Calculate accuracy (for classification only).
        # outputs.data.shape[-1] > 1 distinguishes true multi-class (C logits)
        # from regression with a single output neuron (shape (N,1)), which would
        # otherwise enter this branch and produce argmax=0 for every sample.
        if len(outputs.data.shape) > 1 and outputs.data.shape[-1] > 1:  # Multi-class
            predictions = np.argmax(outputs.data, axis=1)
            if len(targets.data.shape) == 1:  # Integer targets
                correct += np.sum(predictions == targets.data)
            else:  # One-hot targets
                correct += np.sum(predictions == np.argmax(targets.data, axis=1))
            total += len(predictions)

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0

    self.history['eval_loss'].append(avg_loss)

    return avg_loss, accuracy
    

Trainer.evaluate = trainer_evaluate

###  Unit Test: Trainer.evaluate

In [37]:
def test_unit_trainer_evaluate():
    """ Test Trainer.evaluate implementation."""
    print(" Unit Test: Trainer.evaluate...")

    # --- Regression case: output shape (N, 1) must NOT enter the accuracy branch ---
    class RegressionModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    reg_model = RegressionModel()
    reg_trainer = Trainer(reg_model, SGD(reg_model.parameters(), lr=0.01), MSELoss())

    reg_dataloader = [
        (Tensor([[1.0, 0.5]]), Tensor([[2.0]])),
        (Tensor([[0.5, 1.0]]), Tensor([[1.5]]))
    ]

    eval_loss, accuracy = reg_trainer.evaluate(reg_dataloader)

    assert isinstance(eval_loss, (float, np.floating)), f"Expected float eval_loss, got {type(eval_loss)}"
    assert isinstance(accuracy, (float, np.floating)), f"Expected float accuracy, got {type(accuracy)}"
    # Regression: accuracy must be 0.0 (no classification branch entered)
    assert accuracy == 0.0, (
        f"Regression evaluate() returned accuracy={accuracy:.4f} instead of 0.0. "
        "Shape (N,1) outputs must not enter the argmax classification branch."
    )

    assert reg_trainer.training_mode is False, "Should be in eval mode after evaluate()"
    assert reg_model.training is False, "Model should be in eval mode"
    assert len(reg_trainer.history['eval_loss']) == 1, "Should have 1 eval loss recorded"
    assert np.isfinite(eval_loss), f"Eval loss should be finite, got {eval_loss}"

    # --- Classification case: output shape (N, C) with C > 1 must compute accuracy ---
    class ClassificationModel:
        def __init__(self):
            self.layer = Linear(2, 3)  # 3-class output
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    cls_model = ClassificationModel()
    cls_trainer = Trainer(cls_model, SGD(cls_model.parameters(), lr=0.01), MSELoss())

    cls_dataloader = [
        (Tensor([[1.0, 0.5]]), Tensor([0])),   # integer class label
        (Tensor([[0.5, 1.0]]), Tensor([2]))
    ]

    _, cls_accuracy = cls_trainer.evaluate(cls_dataloader)
    assert isinstance(cls_accuracy, (float, np.floating)), "Classification accuracy must be a float"
    # accuracy is 0.0 or 1.0 here -- just verify it was actually computed (total > 0)
    assert 0.0 <= cls_accuracy <= 1.0, f"Classification accuracy out of range: {cls_accuracy}"

    print(f"  Regression  - loss: {eval_loss:.4f}, accuracy: {accuracy:.4f} (expected 0.0)")
    print(f"  Classification - accuracy: {cls_accuracy:.4f} (range check passed)")
    print("✅ Trainer.evaluate works correctly!")

if __name__ == "__main__":
    test_unit_trainer_evaluate()

 Unit Test: Trainer.evaluate...
  Regression  - loss: 3.5819, accuracy: 0.0000 (expected 0.0)
  Classification - accuracy: 1.0000 (range check passed)
✅ Trainer.evaluate works correctly!


###  Trainer.save_checkpoint - Persisting Training State

Checkpointing saves everything needed to resume training later: model weights,
optimizer state, scheduler state, epoch count, and training history. This is
essential for long training runs that may be interrupted.

In [38]:
#| export
def trainer_save_checkpoint(self, path: str):
    """
    Save complete training state for resumption.
    """
    
    checkpoint = {
        'epoch': self.epoch,
        'step': self.step,
        'model_state': self._get_model_state(),
        'optimizer_state': self._get_optimizer_state(),
        'scheduler_state': self._get_scheduler_state(),
        'history': self.history,
        'training_mode': self.training_mode
    }

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(checkpoint, f)
    

Trainer.save_checkpoint = trainer_save_checkpoint

###  Unit Test: Trainer.save_checkpoint

In [39]:
def test_unit_trainer_save_checkpoint():
    """ Test Trainer.save_checkpoint implementation."""
    print(" Unit Test: Trainer.save_checkpoint...")

    class DummyModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    model = DummyModel()
    optimizer = SGD(model.parameters(), lr=0.05)
    trainer = Trainer(model, optimizer, MSELoss())

    # Set some state to verify it persists
    trainer.epoch = 5
    trainer.step = 100
    trainer.history['train_loss'].append(0.5)

    checkpoint_path = "/tmp/test_save_checkpoint.pkl"
    trainer.save_checkpoint(checkpoint_path)

    # Verify file exists
    assert os.path.exists(checkpoint_path), "Checkpoint file should exist"

    # Verify contents
    with open(checkpoint_path, 'rb') as f:
        checkpoint = pickle.load(f)

    assert checkpoint['epoch'] == 5, f"Expected epoch=5, got {checkpoint['epoch']}"
    assert checkpoint['step'] == 100, f"Expected step=100, got {checkpoint['step']}"
    assert 'model_state' in checkpoint, "Missing model_state"
    assert 'optimizer_state' in checkpoint, "Missing optimizer_state"
    assert 'history' in checkpoint, "Missing history"
    assert len(checkpoint['history']['train_loss']) == 1, "History should have 1 entry"

    # Clean up
    os.remove(checkpoint_path)

    print("✅ Trainer.save_checkpoint works correctly!")

if __name__ == "__main__":
    test_unit_trainer_save_checkpoint()

 Unit Test: Trainer.save_checkpoint...
✅ Trainer.save_checkpoint works correctly!


###  Trainer.load_checkpoint - Resuming Training

Loading a checkpoint restores the exact training state so we can continue
where we left off. This means restoring epoch count, optimizer state
(including momentum buffers), and the full training history.

In [40]:
#| export
def trainer_load_checkpoint(self, path: str):
    """
    Load training state from checkpoint.
    """
    
    with open(path, 'rb') as f:
        checkpoint = pickle.load(f)

    self.epoch = checkpoint['epoch']
    self.step = checkpoint['step']
    self.history = checkpoint['history']
    self.training_mode = checkpoint['training_mode']

    # Restore states
    if 'model_state' in checkpoint:
        self._set_model_state(checkpoint['model_state'])
    if 'optimizer_state' in checkpoint:
        self._set_optimizer_state(checkpoint['optimizer_state'])
    if 'scheduler_state' in checkpoint:
        self._set_scheduler_state(checkpoint['scheduler_state'])
    

Trainer.load_checkpoint = trainer_load_checkpoint

###  Unit Test: Trainer.load_checkpoint

In [41]:
def test_unit_trainer_load_checkpoint():
    """ Test Trainer.load_checkpoint implementation."""
    print(" Unit Test: Trainer.load_checkpoint...")

    class DummyModel:
        def __init__(self):
            self.layer = Linear(2, 1)
            self.training = True
        def forward(self, x):
            return self.layer.forward(x)
        def parameters(self):
            return self.layer.parameters()

    model = DummyModel()
    optimizer = SGD(model.parameters(), lr=0.05)
    trainer = Trainer(model, optimizer, MSELoss())

    # Set distinctive state
    trainer.epoch = 7
    trainer.step = 200
    trainer.history['train_loss'].extend([0.9, 0.7, 0.5])
    trainer.training_mode = False

    # Save original model weights for comparison
    original_weights = model.parameters()[0].data.copy()

    checkpoint_path = "/tmp/test_load_checkpoint.pkl"
    trainer.save_checkpoint(checkpoint_path)

    # Corrupt state
    trainer.epoch = 999
    trainer.step = 0
    trainer.history = {'train_loss': [], 'eval_loss': [], 'learning_rates': []}
    trainer.training_mode = True

    # Restore
    trainer.load_checkpoint(checkpoint_path)

    # Verify restoration
    assert trainer.epoch == 7, f"Expected epoch=7, got {trainer.epoch}"
    assert trainer.step == 200, f"Expected step=200, got {trainer.step}"
    assert trainer.training_mode is False, "training_mode should be restored to False"
    assert len(trainer.history['train_loss']) == 3, "History should have 3 entries"
    assert trainer.history['train_loss'] == [0.9, 0.7, 0.5], "History values should match"

    # Verify model weights restored
    restored_weights = model.parameters()[0].data
    assert np.allclose(restored_weights, original_weights), "Model weights should be restored"

    # Clean up
    os.remove(checkpoint_path)

    print("✅ Trainer.load_checkpoint works correctly!")

if __name__ == "__main__":
    test_unit_trainer_load_checkpoint()

 Unit Test: Trainer.load_checkpoint...
✅ Trainer.load_checkpoint works correctly!


##  Integration: Complete Training Example

Now let's create a complete training example that demonstrates how all the components work together.

In [42]:
def demonstrate_complete_training_pipeline():
    """
    Complete end-to-end training example using all components..
    """
    print(" Building Complete Training Pipeline...")
    print("=" * 60)

    # Step 1: Create model using REAL Linear layer
    class SimpleNN:
        def __init__(self):
            self.layer1 = Linear(3, 5)
            self.layer2 = Linear(5, 2)
            self.training = True

        def forward(self, x):
            x = self.layer1.forward(x)
            # Simple ReLU-like activation (max with 0)
            x = Tensor(np.maximum(0, x.data))
            x = self.layer2.forward(x)
            return x

        def parameters(self):
            return self.layer1.parameters() + self.layer2.parameters()

    print(" Model created: 3 → 5 → 2 network")

    # Step 2: Create optimizer
    model = SimpleNN()
    optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9)
    print(" Optimizer: SGD with momentum")

    # Step 3: Create loss function
    loss_fn = MSELoss()
    print(" Loss function: MSE")

    # Step 4: Create scheduler
    scheduler = CosineSchedule(max_lr=0.1, min_lr=0.001, total_epochs=5)
    print(" Scheduler: Cosine annealing (0.1 → 0.001)")

    # Step 5: Create trainer with gradient clipping
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scheduler=scheduler,
        grad_clip_norm=1.0
    )
    print(" Trainer initialized with gradient clipping")

    # Step 6: Create synthetic training data
    train_data = [
        (Tensor(rng.standard_normal((4, 3))), Tensor(rng.standard_normal((4, 2)))),
        (Tensor(rng.standard_normal((4, 3))), Tensor(rng.standard_normal((4, 2)))),
        (Tensor(rng.standard_normal((4, 3))), Tensor(rng.standard_normal((4, 2))))
    ]
    print(" Training data: 3 batches of 4 samples")

    # Step 7: Train for multiple epochs
    print("\n🚀 Starting Training...")
    print("-" * 60)
    print(f"{'Epoch':<8} {'Train Loss':<12} {'Learning Rate':<15}")
    print("-" * 60)

    for epoch in range(3):
        loss = trainer.train_epoch(train_data)
        lr = scheduler.get_lr(epoch)
        print(f"{epoch:<8} {loss:<12.6f} {lr:<15.6f}")

    # Step 8: Save checkpoint
    checkpoint_path = "/tmp/training_example_checkpoint.pkl"
    trainer.save_checkpoint(checkpoint_path)
    print(f"\n✓ Checkpoint saved: {checkpoint_path}")

    # Step 9: Evaluate
    eval_loss, accuracy = trainer.evaluate(train_data)
    print(f"✓ Evaluation - Loss: {eval_loss:.6f}, Accuracy: {accuracy:.6f}")

    # Clean up
    import os
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print("\n" + "=" * 60)
    print(" Complete training pipeline executed successfully!")

##  Systems Analysis: Training Performance and Memory


In [44]:
def analyze_training_memory():
    """ Analyze memory overhead of training components."""
    print(" Analyzing Training Memory Overhead...")

    # Create models of different sizes
    model_sizes = [
        ("Small", 100),    # 100 parameters
        ("Medium", 1000),  # 1K parameters
        ("Large", 10000)   # 10K parameters
    ]

    print("\nTraining Memory Analysis:")
    print("=" * 90)
    print(f"{'Model':<10} {'Params':<10} {'Gradients':<12} {'SGD State':<12} {'Adam State':<12} {'Total':<10}")
    print("-" * 90)

    for name, param_count in model_sizes:
        # Base memory: parameters
        param_memory = param_count * 4  # 4 bytes per float32

        # Gradients: same as parameters
        grad_memory = param_count * 4

        # SGD optimizer state: momentum buffer
        sgd_memory = param_count * 4

        # Adam optimizer state: 2 buffers (m and v)
        adam_memory = param_count * 2 * 4

        # Total with Adam (worst case)
        total_memory = param_memory + grad_memory + adam_memory

        # Convert to human-readable
        def format_memory(bytes):
            if bytes < 1024:
                return f"{bytes}B"
            elif bytes < 1024 * 1024:
                return f"{bytes/1024:.1f}KB"
            else:
                return f"{bytes/(1024*1024):.1f}MB"

        print(f"{name:<10} {format_memory(param_memory):<10} "
              f"{format_memory(grad_memory):<12} {format_memory(sgd_memory):<12} "
              f"{format_memory(adam_memory):<12} {format_memory(total_memory):<10}")

    print("\n💡 Key Insights:")
    print("- Training memory = Parameters + Gradients + Optimizer State")
    print("- SGD: 3× parameter memory (params + grads + momentum)")
    print("- Adam: 4× parameter memory (params + grads + 2 moment buffers)")
    print("- Gradient accumulation reduces memory but increases training time")

In [46]:
def analyze_checkpoint_overhead():
    """ Analyze checkpoint size and overhead."""
    print("\n Analyzing Checkpoint Overhead...")

    # Create a simple model
    class TinyModel:
        def __init__(self, size):
            self.layer = Linear(size, size)
            self.training = True

        def forward(self, x):
            return self.layer.forward(x)

        def parameters(self):
            return self.layer.parameters()

    sizes = [10, 50, 100]

    print("\nCheckpoint Size Analysis:")
    print("=" * 70)
    print(f"{'Model Size':<12} {'Raw Params':<15} {'Checkpoint':<15} {'Overhead':<10}")
    print("-" * 70)

    import pickle
    import sys

    for size in sizes:
        # Create model and trainer
        model = TinyModel(size)
        optimizer = SGD(model.parameters(), lr=0.01)
        trainer = Trainer(model, optimizer, MSELoss())

        # Estimate raw parameter size
        param_count = size * size + size  # W + b
        raw_size = param_count * 4  # 4 bytes per float32

        # Create checkpoint and measure size
        checkpoint_path = f"/tmp/checkpoint_test_{size}.pkl"
        trainer.save_checkpoint(checkpoint_path)

        import os
        checkpoint_size = os.path.getsize(checkpoint_path)
        overhead = (checkpoint_size / raw_size - 1) * 100

        # Clean up
        os.remove(checkpoint_path)

        def format_size(bytes):
            if bytes < 1024:
                return f"{bytes}B"
            return f"{bytes/1024:.1f}KB"

        print(f"{size}×{size:<8} {format_size(raw_size):<15} "
              f"{format_size(checkpoint_size):<15} {overhead:.1f}%")

    print("\n💡 Key Insights:")
    print("- Checkpoints include model state + optimizer state + training metadata")
    print("- Pickle serialization adds 10-30% overhead")
    print("- Adam optimizer doubles checkpoint size vs SGD")

# Run the systems analysis
if __name__ == "__main__":
    analyze_training_memory()
    analyze_checkpoint_overhead()

 Analyzing Training Memory Overhead...

Training Memory Analysis:
Model      Params     Gradients    SGD State    Adam State   Total     
------------------------------------------------------------------------------------------
Small      400B       400B         400B         800B         1.6KB     
Medium     3.9KB      3.9KB        3.9KB        7.8KB        15.6KB    
Large      39.1KB     39.1KB       39.1KB       78.1KB       156.2KB   

💡 Key Insights:
- Training memory = Parameters + Gradients + Optimizer State
- SGD: 3× parameter memory (params + grads + momentum)
- Adam: 4× parameter memory (params + grads + 2 moment buffers)
- Gradient accumulation reduces memory but increases training time

 Analyzing Checkpoint Overhead...

Checkpoint Size Analysis:
Model Size   Raw Params      Checkpoint      Overhead  
----------------------------------------------------------------------
10×10       440B            772B            75.5%
50×50       10.0KB          10.3KB          3.3%
100

In [47]:
def test_module():
    """ Module Test: Complete Integration

    Comprehensive test of entire module functionality.
    """
    print(" RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_cosine_schedule()
    test_unit_clip_grad_norm()
    test_unit_trainer_init()
    test_unit_trainer_process_batch()
    test_unit_trainer_optimizer_update()
    test_unit_trainer_train_epoch()
    test_unit_trainer_evaluate()
    test_unit_trainer_save_checkpoint()
    test_unit_trainer_load_checkpoint()

    print("\nRunning integration scenarios...")

    # Test complete training pipeline integration with REAL components
    print(" Integration Test: Complete Training Pipeline...")

    # Use REAL components from previous modules (already imported at module level)

    # Create a simple model using REAL Linear layer
    class SimpleModel:
        def __init__(self):
            self.layer = Linear(2, 1)  # Real Linear from Module 03
            self.training = True

        def forward(self, x):
            return self.layer.forward(x)

        def parameters(self):
            return self.layer.parameters()

    # Create integrated system with REAL components
    model = SimpleModel()
    optimizer = SGD(model.parameters(), lr=0.01)  # Real SGD from Module 07
    loss_fn = MSELoss()  # Real MSELoss from Module 04
    scheduler = CosineSchedule(max_lr=0.1, min_lr=0.001, total_epochs=3)

    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scheduler=scheduler,
        grad_clip_norm=0.5
    )

    
    data = [
        (Tensor([[1.0, 0.5]]), Tensor([[0.8]])),
        (Tensor([[0.5, 1.0]]), Tensor([[0.2]]))
    ]

    # Test training
    initial_loss = trainer.train_epoch(data)
    assert isinstance(initial_loss, (float, np.floating)), "Training should return float loss"
    assert trainer.epoch == 1, "Epoch should increment"

    # Test evaluation
    eval_loss, accuracy = trainer.evaluate(data)
    assert isinstance(eval_loss, (float, np.floating)), "Evaluation should return float loss"
    assert isinstance(accuracy, (float, np.floating)), "Evaluation should return float accuracy"

    # Test scheduling
    lr_epoch_0 = scheduler.get_lr(0)
    lr_epoch_1 = scheduler.get_lr(1)
    assert lr_epoch_0 > lr_epoch_1, "Learning rate should decrease"

    # Test gradient clipping with large gradients using real Tensor
    large_param = Tensor([1.0, 2.0], requires_grad=True)
    large_param.grad = np.array([100.0, 200.0])
    large_params = [large_param]

    original_norm = clip_grad_norm(large_params, max_norm=1.0)
    assert original_norm > 1.0, "Original norm should be large"

    if isinstance(large_params[0].grad, np.ndarray):
        grad_data = large_params[0].grad
    else:
        # Trust that Tensor has .data attribute
        grad_data = large_params[0].grad.data
    new_norm = np.linalg.norm(grad_data)
    assert abs(new_norm - 1.0) < 1e-6, "Clipped norm should equal max_norm"

    # Test checkpointing
    checkpoint_path = "/tmp/integration_test_checkpoint.pkl"
    trainer.save_checkpoint(checkpoint_path)

    original_epoch = trainer.epoch
    trainer.epoch = 999
    trainer.load_checkpoint(checkpoint_path)

    assert trainer.epoch == original_epoch, "Checkpoint should restore state"

    # Clean up
    import os
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print("✅ End-to-end training pipeline works!")

    print("\n" + "=" * 50)

if __name__ == "__main__":
    test_module()

 RUNNING MODULE INTEGRATION TEST
Running unit tests...
 Unit Test: CosineSchedule...
Learning rate at epoch 0: 0.1000
Learning rate at epoch 50: 0.0550
Learning rate at epoch 100: 0.0100
 CosineSchedule works correctly!
 Unit Test: Gradient Clipping...
Original norm: 11.18
Clipped norm: 1.00
✅ Gradient clipping works correctly!
 Unit Test: Trainer.__init__...
✅ Trainer.__init__ works correctly!
 Unit Test: Trainer._process_batch...
✅ Trainer._process_batch works correctly!
 Unit Test: Trainer._optimizer_update...
✅ Trainer._optimizer_update works correctly!
 Unit Test: Trainer.train_epoch...
  Epoch 1 loss: 1.9357
  Epoch 2 loss: 1.6652
✅ Trainer.train_epoch works correctly!
 Unit Test: Trainer.evaluate...
  Regression  - loss: 4.2564, accuracy: 0.0000 (expected 0.0)
  Classification - accuracy: 0.0000 (range check passed)
✅ Trainer.evaluate works correctly!
 Unit Test: Trainer.save_checkpoint...
✅ Trainer.save_checkpoint works correctly!
 Unit Test: Trainer.load_checkpoint...
✅ Traine

In [48]:
def demo_training():
    """ See the training loop in action."""
    print(" AHA MOMENT: Training Just Works")
    print("=" * 45)

    # Simple linear regression: learn y = 2x + 1
    rng = np.random.default_rng(7)
    X = Tensor(rng.standard_normal((20, 1)))
    y = Tensor(X.data * 2 + 1)  # True relationship

    # Simple model: one weight, one bias
    w = Tensor(np.array([[0.0]]), requires_grad=True)
    b = Tensor(np.array([0.0]), requires_grad=True)

    optimizer = SGD([w, b], lr=0.1)
    loss_fn = MSELoss()

    print("Learning y = 2x + 1:")
    for epoch in range(5):
        # Forward
        pred = X.matmul(w) + b
        loss = loss_fn(pred, y)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"  Epoch {epoch+1}: w={w.data[0,0]:.2f}, b={b.data[0]:.2f}, loss={float(loss.data):.4f}")

    print(f"\nLearned: y = {w.data[0,0]:.1f}x + {b.data[0]:.1f}")
    print("Target:  y = 2.0x + 1.0")

    print("\n
    Your training loop learned the pattern!")

In [49]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_training()

 RUNNING MODULE INTEGRATION TEST
Running unit tests...
 Unit Test: CosineSchedule...
Learning rate at epoch 0: 0.1000
Learning rate at epoch 50: 0.0550
Learning rate at epoch 100: 0.0100
 CosineSchedule works correctly!
 Unit Test: Gradient Clipping...
Original norm: 11.18
Clipped norm: 1.00
✅ Gradient clipping works correctly!
 Unit Test: Trainer.__init__...
✅ Trainer.__init__ works correctly!
 Unit Test: Trainer._process_batch...
✅ Trainer._process_batch works correctly!
 Unit Test: Trainer._optimizer_update...
✅ Trainer._optimizer_update works correctly!
 Unit Test: Trainer.train_epoch...
  Epoch 1 loss: 4.2268
  Epoch 2 loss: 3.5638
✅ Trainer.train_epoch works correctly!
 Unit Test: Trainer.evaluate...
  Regression  - loss: 1.2061, accuracy: 0.0000 (expected 0.0)
  Classification - accuracy: 0.0000 (range check passed)
✅ Trainer.evaluate works correctly!
 Unit Test: Trainer.save_checkpoint...
✅ Trainer.save_checkpoint works correctly!
 Unit Test: Trainer.load_checkpoint...
✅ Traine

In [50]:
from nbdev.export import nb_export
nb_export("training.ipynb", lib_path='../tinytorch')